[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Object-Oriented Python](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)

# Composition over Inheritance &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.


**1.** A class that holds a list.


In [1]:
class Log:
    def __init__(self):
        self._messages = []

    def add(self, message):
        self._messages.append(message)

    def __len__(self):
        return len(self._messages)


log = Log()
log.add("Tromso: sensor restarted")
log.add("Tromso: reading taken")

print(len(log))


2


`__len__` passes the question to the list it holds. Without it, `len(log)` would raise, because
holding a list does not make `Log` one.


**2.** Making it loopable.


In [2]:
class Log:
    def __init__(self):
        self._messages = []

    def add(self, message):
        self._messages.append(message)

    def __len__(self):
        return len(self._messages)

    def __iter__(self):
        return iter(self._messages)


log = Log()
log.add("Tromso: sensor restarted")
log.add("Tromso: reading taken")

for message in log:
    print(message)


Tromso: sensor restarted
Tromso: reading taken


`__iter__` hands back the list's own iterator, the shortest correct version from the **Context
Managers and Iterators** notebook.


**3.** The methods `Log` chose not to have.


In [3]:
try:
    log.clear()
except AttributeError as error:
    print("AttributeError:", error)

try:
    log[0] = "x"
except TypeError as error:
    print("TypeError:", error)

# Both errors are the point. A log is a record of what happened, so clearing it
# or overwriting an entry would destroy that record. Because Log holds its list
# rather than inheriting from list, it never offered either operation, and there
# is no way to do either by accident.
print(len(log), "messages, both still there")


AttributeError: 'Log' object has no attribute 'clear'
TypeError: 'Log' object does not support item assignment
2 messages, both still there


Every operation `Log` does not offer is one that cannot be misused. With `class Log(list)`, both of
these would have worked silently.


**4.** A station that holds a log.


In [4]:
class Station:
    def __init__(self, name):
        self.name = name
        self.log = Log()

    def note(self, text):
        self.log.add(f"{self.name}: {text}")


north = Station("Tromso")
north.note("sensor restarted")
north.note("reading taken")

for message in north.log:
    print(message)


Tromso: sensor restarted
Tromso: reading taken


`note` adds the station's name and passes the rest to the log. The station decides what goes in a
message, and the log decides how messages are kept, and neither needs to know how the other works.


**5.** A formatter that can be swapped.


In [5]:
class Celsius:
    def show(self, value):
        return f"{value:.1f} C"


class Fahrenheit:
    def show(self, value):
        return f"{value * 9 / 5 + 32:.1f} F"


class Station:
    def __init__(self, name, readings, formatter):
        self.name = name
        self.readings = readings
        self.formatter = formatter

    def report(self):
        mean = round(sum(self.readings) / len(self.readings), 2)
        return f"{self.name}: mean {self.formatter.show(mean)}"


north = Station("Tromso", [-4.1, -2.6], Celsius())
print(north.report())

north.formatter = Fahrenheit()
print(north.report())


Tromso: mean -3.4 C
Tromso: mean 26.0 F


The same object reports in a different unit, because one of its parts was replaced. Nothing about
`Station` had to change, and a third formatter would need no change to it either.


**6.** Inherit or hold.


In [6]:
# UrgentAlert and Alert: inherit. An urgent alert is a kind of alert, and every
# Alert method applies to it, so it passes both questions.

# Inventory and dict: hold. An inventory has items, and wanting one method is
# the sign of borrowing; a dict would also bring clear, popitem and the rest.

# Timer and Station: hold, or neither. A timer is not a kind of station, so
# reusing report is borrowing. If both need the same formatting, that belongs
# in a part they can each hold.
print("nothing to run for this one")


nothing to run for this one


The first pair is the only genuine kind. The other two share a symptom: one class wants a single
method from another, which is borrowing, and composition is the tool for it.


---

&#8592; **Back to:** [Composition over Inheritance](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/10-composition-over-inheritance.ipynb)  &nbsp;&middot;&nbsp;  [Object-Oriented Python Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)
